# LSTM

Owner: _(name)_

Rules shared by all three model notebooks (brief, sections 3 to 5):

- read the data and the split from `src` (never re-download, never re-split);
- fit on train, choose settings on validation, touch test once at the end;
- state the rolling forecast procedure: each forecast uses only information available at that time;
- compare several plausible configurations and keep a table of settings and validation results;
- time the training (computational cost) and save test predictions with
  `common.save_predictions("lstm", df, predictions, settings=..., train_seconds=...)`, which
  converts them to index points for part 3.

## How an LSTM works, in plain words (for the report's methods section)

An LSTM is a recurrent network whose memory cell is protected by three gates, each a small learned
switch between 0 and 1: the **forget gate** decides how much of the old memory to keep, the **input
gate** decides how much of today's information to write in, and the **output gate** decides how much of
the memory to show to the next layer. The gates let the network keep information for many days when
useful and drop it when not, which a vanilla RNN cannot do reliably.

### Setup (same cell in every notebook)

Runs in Colab and in Cursor. In Colab it clones the repository first; for a private repository
paste a GitHub token when asked (Settings > Developer settings > Fine-grained tokens, read access
to this repository is enough).

In [ ]:
import sys, pathlib

REPO_URL = "https://github.com/Makkaii7/sp500-forecasting.git"   # <- set once after the repo exists

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and not pathlib.Path("sp500-forecasting").exists() and not pathlib.Path("src/config.py").exists():
    from getpass import getpass
    token = getpass("GitHub token (leave empty if the repo is public): ")
    url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL
    import subprocess
    subprocess.run(["git", "clone", "--quiet", url], check=True)
if IN_COLAB and pathlib.Path("sp500-forecasting").exists():
    import os
    os.chdir("sp500-forecasting")

root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "config.py").exists())
sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src import config, common

df = common.load_prices()
y = common.target_series(df)
train, val, test = common.split(y)
print(root)
print(df.index.min().date(), "to", df.index.max().date(), "|", len(df), "trading days")
print(f"train {train.index.min().date()} to {train.index.max().date()} ({len(train)}) | validation {val.index.min().date()} to {val.index.max().date()} ({len(val)}) | test {test.index.min().date()} to {test.index.max().date()} ({len(test)})")
df.tail(3)

## 1. Windows of the last `config.WINDOW` days

The scaler is fitted on the training period only. Each input is the W previous values; the target is
the next value. The history for the first validation and test windows comes from the days before them,
which were known at the time, so every forecast uses only past information.

In [ ]:
W = config.WINDOW
from sklearn.preprocessing import MinMaxScaler

def make_windows(scaled_values, prefix, w):
    full = np.concatenate([prefix, scaled_values])
    X, Y = [], []
    for i in range(len(scaled_values)):
        j = len(prefix) + i
        X.append(full[j - w:j]); Y.append(full[j])
    return np.array(X)[..., None], np.array(Y)

def prepare(w):
    scaler = MinMaxScaler(feature_range=(-1, 1)).fit(train.to_numpy().reshape(-1, 1))
    sc = lambda s: scaler.transform(s.to_numpy().reshape(-1, 1)).ravel()
    tr, va, te = sc(train), sc(val), sc(test)
    X_tr, Y_tr = make_windows(tr[w:], tr[:w], w)
    X_va, Y_va = make_windows(va, tr[-w:], w)
    X_te, Y_te = make_windows(te, np.concatenate([tr, va])[-w:], w)
    return scaler, (X_tr, Y_tr), (X_va, Y_va), (X_te, Y_te)

scaler, (X_train, Y_train), (X_val, Y_val), (X_test, Y_test) = prepare(W)
X_train.shape, X_val.shape, X_test.shape

## 2. The network, as a function of its settings

In [ ]:
import keras
from keras import layers

def build(units=32, window=W, dropout=0.1, learning_rate=1e-3):
    keras.utils.set_random_seed(config.RANDOM_SEED)
    model = keras.Sequential([
        keras.Input(shape=(window, 1)),
        layers.LSTM(units),
        layers.Dropout(dropout),
        layers.Dense(1),                  # one number out, no activation: a return can be any value
    ])
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=learning_rate), loss="mean_squared_error", metrics=["mean_absolute_error"])
    return model

build().summary()

## 3. Compare plausible configurations on validation (brief, section 4)

Each row: settings, validation RMSE in index points, training time. Early stopping on the validation
loss is the overfitting control; the epoch it stopped at is recorded too.

In [ ]:
def unscale(scaled, scaler):
    return scaler.inverse_transform(np.asarray(scaled).reshape(-1, 1)).ravel()

grid = [dict(units=16, window=10), dict(units=32, window=20), dict(units=64, window=20), dict(units=32, window=40)]
rows, fitted = [], {}
for g in grid:
    scaler_g, (Xtr, Ytr), (Xva, Yva), (Xte, Yte) = prepare(g["window"])
    model = build(units=g["units"], window=g["window"])
    stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
    with common.Stopwatch() as sw:
        hist = model.fit(Xtr, Ytr, epochs=60, batch_size=32, validation_data=(Xva, Yva), callbacks=[stop], verbose=0)
    val_pred = common.to_price(pd.Series(unscale(model.predict(Xva, verbose=0), scaler_g), index=val.index), df)
    rows.append({**g, "epochs run": len(hist.history["loss"]), "val RMSE (index points)": common.rmse(common.actual_price(val.index, df), val_pred), "train seconds": sw.seconds})
    fitted[(g["units"], g["window"])] = (model, scaler_g, (Xte, Yte))
configs = pd.DataFrame(rows).round(3)
configs

## 4. The chosen configuration: test once, in index points, and save

In [ ]:
best = configs.sort_values("val RMSE (index points)").iloc[0]
best = {k: (v.item() if hasattr(v, "item") else v) for k, v in best.items()}      # plain Python numbers
model, scaler_b, (Xte, Yte) = fitted[(int(best["units"]), int(best["window"]))]
test_pred = pd.Series(unscale(model.predict(Xte, verbose=0), scaler_b), index=test.index)
test_price = common.to_price(test_pred, df)
print("chosen:", best)
print("test MAE  (index points)", round(common.mae(common.actual_price(test.index, df), test_price), 2))
print("test RMSE (index points)", round(common.rmse(common.actual_price(test.index, df), test_price), 2))
common.save_predictions("lstm", df, test_pred, settings=best, train_seconds=float(best["train seconds"]))

## 5. Learning curve of the chosen model (overfitting check)

Re-fit the chosen settings without early stopping if you want the full curve; otherwise describe the
curve from the run above: where the validation loss stopped improving, and the gap to training loss.